In [1]:
!pip install autogluon pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [2]:
import pandas as pd
import numpy as np

from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('/content/df_all_clean (1).csv')

print(df.shape)
df.head()

(19350, 25)


,discode,Sex,agey,agem,aged,nation,occupat,addercode,metropol,treatmentloc,...,datefound,datedeath,germ,complications,district,province,death,days_to_death,days_to_define,days_to_report
0,66.0,หญิง,17,5,19,ไทย,ไม่ทราบอาชีพ,26020302,อบต,ร.พ.ชุมชน,...,2018-01-01,NaN,NaN,NaN,ปากพลี,นครนายก,0,NaN,0.0,0.0
1,66.0,ชาย,52,8,1,ไทย,รับจ้าง-กรรมกร,14081305,ในเขตเทศบาล,ร.พ.ชุมชน,...,2018-01-01,NaN,NaN,NaN,ผักไห่,พระนครศรีอยุธยา,0,NaN,0.0,0.0
2,66.0,ชาย,55,0,0,ไทย,รับจ้าง-กรรมกร,14110303,ในเขตเทศบาล,คลินิก-ร.พ.เอกชน,...,2018-01-01,NaN,9.0,0.0,วังน้อย,พระนครศรีอยุธยา,0,NaN,0.0,0.0
3,66.0,ชาย,39,0,0,ไทย,รับจ้าง-กรรมกร,14100605,อบต,ร.พ.ชุมชน,...,2018-01-01,NaN,0.0,0.0,ลาดบัวหลวง,พระนครศรีอยุธยา,0,NaN,0.0,0.0
4,66.0,ชาย,51,3,24,ไทย,รับจ้าง-กรรมกร,14051102,ในเขตเทศบาล,ร.พ.ชุมชน,...,2018-01-01,NaN,NaN,NaN,บางบาล,พระนครศรีอยุธยา,0,NaN,0.0,0.0


In [29]:
# change columns to datetime

df["datesick"] = pd.to_datetime(df["datesick"], errors="coerce")

In [31]:
# สร้าง Month Feature

df["month"] = df["datesick"].dt.to_period("M")

In [32]:
# Aggregate Cases

cases = (
    df.groupby(["province","district","month"])
    .size()
    .reset_index(name="cases")
)

In [7]:
# แปลง Month เป็น Datetime

cases["month"] = cases["month"].dt.to_timestamp()

In [8]:
# เรียงข้อมูลตามเวลา

cases = cases.sort_values(
    ["province","district","month"]
)

In [9]:
# Lag Feature (1 เดือนก่อน)

cases["cases_lag1"] = (
    cases.groupby(["province","district"])["cases"]
    .shift(1)
)

In [10]:
# Lag Feature (2 เดือนก่อน)

cases["cases_lag2"] = (
    cases.groupby(["province","district"])["cases"]
    .shift(2)
)

In [11]:
# Lag Feature (3 เดือนก่อน)

cases["cases_lag3"] = (
    cases.groupby(["province","district"])["cases"]
    .shift(3)
)

In [12]:
# Trend Feature

cases["trend"] = cases["cases_lag1"] - cases["cases_lag2"]

In [13]:
# Acceleration Feature

cases["acceleration"] = (
    cases["cases_lag1"]
    - 2*cases["cases_lag2"]
    + cases["cases_lag3"]
)

In [14]:
# Moving Average (3 เดือน)

cases["ma3"] = (
    cases.groupby(["province","district"])["cases"]
    .rolling(3)
    .mean()
    .reset_index(level=[0,1], drop=True)
)

In [15]:
# Moving Average (6 เดือน)

cases["ma6"] = (
    cases.groupby(["province","district"])["cases"]
    .rolling(6)
    .mean()
    .reset_index(level=[0,1], drop=True)
)

In [16]:
# Growth Rate

cases["growth_rate"] = (
    cases["cases_lag1"] - cases["cases_lag2"]
) / (cases["cases_lag2"] + 1)

In [17]:
# Seasonality

cases["month_num"] = cases["month"].dt.month

In [18]:
# กำหนด Hotspot

threshold = 30

cases["hotspot"] = (
    cases["cases"] > threshold
).astype(int)

In [19]:
# ลบ Missing Data

cases = cases.dropna()

In [20]:
# เลือก Feature

features = [
"province",
"district",
"cases_lag1",
"cases_lag2",
"cases_lag3",
"trend",
"acceleration",
"ma3",
"ma6",
"growth_rate",
"month_num",
"hotspot"
]

In [21]:
# เตรียม Dataset

dataset = cases[features]

In [22]:
# Split Dataset

train_data, test_data = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42
)

In [23]:
# Train Model

predictor = TabularPredictor(
    label="hotspot",
    problem_type="binary",
    eval_metric="roc_auc"
).fit(
    train_data,
    presets="high_quality"
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260315_075816"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.15 GB / 12.67 GB (88.0%)
Disk Space Avail:   74.75 GB / 107.72 GB (69.4%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if m

(_ray_fit pid=3021) [1000]	valid_set's binary_logloss: 0.0120697
(_ray_fit pid=3021) [2000]	valid_set's binary_logloss: 0.0090854


(_dystack pid=2796) 	0.9905	 = Validation score   (roc_auc)
(_dystack pid=2796) 	38.14s	 = Training   runtime
(_dystack pid=2796) 	0.18s	 = Validation runtime
(_dystack pid=2796) Fitting model: LightGBM_BAG_L1 ... Training model for up to 525.62s of the 818.62s of remaining time.
(_dystack pid=2796) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.18%)


(_ray_fit pid=3682) [1000]	valid_set's binary_logloss: 0.00694807
(_ray_fit pid=3682) [2000]	valid_set's binary_logloss: 0.00619771
(_ray_fit pid=3682) [3000]	valid_set's binary_logloss: 0.00630908


(_dystack pid=2796) 	0.9895	 = Validation score   (roc_auc)
(_dystack pid=2796) 	34.79s	 = Training   runtime
(_dystack pid=2796) 	0.12s	 = Validation runtime
(_dystack pid=2796) Fitting model: RandomForestGini_BAG_L1 ... Training model for up to 482.38s of the 775.38s of remaining time.
(_dystack pid=2796) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2796) 	0.9799	 = Validation score   (roc_auc)
(_dystack pid=2796) 	2.35s	 = Training   runtime
(_dystack pid=2796) 	0.21s	 = Validation runtime
(_dystack pid=2796) Fitting model: RandomForestEntr_BAG_L1 ... Training model for up to 479.79s of the 772.78s of remaining time.
(_dystack pid=2796) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2796) 	0.9798	 = Validation score   (roc_auc)
(_dystack pid=2796) 	1.14s	 = Training   runtime
(_dystack pid=2796) 	0.17s	 = Validation runtime
(_dystack pid=2796) Fitting model: CatBoost_BAG_L1 ... Training m

(_ray_fit pid=8938) [1000]	valid_set's binary_logloss: 0.000390938


(_dystack pid=2796) 	0.991	 = Validation score   (roc_auc)
(_dystack pid=2796) 	40.22s	 = Training   runtime
(_dystack pid=2796) 	0.11s	 = Validation runtime
(_dystack pid=2796) Fitting model: RandomForestGini_BAG_L2 ... Training model for up to 167.25s of the 167.09s of remaining time.
(_dystack pid=2796) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2796) 	0.996	 = Validation score   (roc_auc)
(_dystack pid=2796) 	2.58s	 = Training   runtime
(_dystack pid=2796) 	0.23s	 = Validation runtime
(_dystack pid=2796) Fitting model: RandomForestEntr_BAG_L2 ... Training model for up to 164.39s of the 164.24s of remaining time.
(_dystack pid=2796) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2796) 	0.988	 = Validation score   (roc_auc)
(_dystack pid=2796) 	1.32s	 = Training   runtime
(_dystack pid=2796) 	0.18s	 = Validation runtime
(_dystack pid=2796) Fitting model: CatBoost_BAG_L2 ... Training mode

In [33]:
# model leaderboard

predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetFastAI_r145_BAG_L1_FULL,0.996097,NaN,roc_auc,0.045008,NaN,4.823346,0.045008,NaN,4.823346,1,True,63
1,CatBoost_r9_BAG_L1_FULL,0.993755,NaN,roc_auc,0.034152,NaN,6.241527,0.034152,NaN,6.241527,1,True,53
2,WeightedEnsemble_L2_FULL,0.993755,NaN,roc_auc,0.154750,NaN,8.533474,0.004782,NaN,0.252979,2,True,74
3,CatBoost_r50_BAG_L1_FULL,0.993365,NaN,roc_auc,0.008967,NaN,0.978161,0.008967,NaN,0.978161,1,True,68
4,NeuralNetFastAI_r102_BAG_L1_FULL,0.992779,NaN,roc_auc,0.075388,NaN,11.307555,0.075388,NaN,11.307555,1,True,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,NeuralNetTorch_r22_BAG_L1,NaN,0.973011,roc_auc,NaN,0.424275,235.105056,NaN,0.424275,235.105056,1,False,18
70,NeuralNetTorch_r79_BAG_L1,NaN,0.965690,roc_auc,NaN,0.368254,146.026797,NaN,0.368254,146.026797,1,False,13
71,LightGBMLarge_BAG_L1,NaN,0.957259,roc_auc,NaN,0.116834,36.362786,NaN,0.116834,36.362786,1,False,11
72,NeuralNetTorch_r86_BAG_L1,NaN,0.950359,roc_auc,NaN,0.294539,126.890838,NaN,0.294539,126.890838,1,False,30


In [25]:
# model prediction

pred = predictor.predict(test_data)

In [26]:
# Probability Prediction

prob = predictor.predict_proba(test_data)

In [27]:
# Feature Importance

predictor.feature_importance(test_data)

Computing feature importance via permutation shuffling for 11 features using 380 rows with 5 shuffle sets...
	10.62s	= Expected runtime (2.12s per shuffle set)
	3.79s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
ma3,0.812217,0.073960,0.000008,5,0.964501,0.659933
cases_lag1,0.008314,0.002783,0.001305,5,0.014043,0.002584
cases_lag2,0.005152,0.003314,0.012714,5,0.011975,-0.001671
growth_rate,0.001717,0.002081,0.069365,5,0.006002,-0.002567
cases_lag3,0.001678,0.000449,0.000561,5,0.002603,0.000753
acceleration,0.001249,0.000509,0.002685,5,0.002297,0.000201
month_num,0.000976,0.000816,0.027827,5,0.002657,-0.000705
province,0.000351,0.000505,0.097465,5,0.001391,-0.000689
ma6,0.000078,0.000223,0.238310,5,0.000536,-0.000380
trend,-0.000078,0.000405,0.655771,5,0.000755,-0.000911


In [28]:
# Save training folder from Colab to Google Drive

from google.colab import drive
import os

# 1) Mount Google Drive
drive.mount('/content/drive')

# 2) กำหนด path ของ folder ที่ต้องการ save
source_folder = "/content/AutogluonModels"   # เปลี่ยนเป็น folder ที่คุณต้องการ
zip_name = "Disease Hotspot Prediction.zip"

# 3) zip folder
!zip -r $zip_name $source_folder

# 4) copy zip file ไป Google Drive
drive_path = "/content/drive/MyDrive/"
!cp $zip_name $drive_path

# 5) แสดงผลลัพธ์
print("Saved to Google Drive:", drive_path + zip_name)

Mounted at /content/drive
	zip warning: name not matched: Hotspot
	zip warning: name not matched: Prediction.zip
  adding: content/AutogluonModels/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_075816/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_075816/learner.pkl (deflated 66%)
  adding: content/AutogluonModels/ag-20260315_075816/version.txt (stored 0%)
  adding: content/AutogluonModels/ag-20260315_075816/predictor.pkl (deflated 50%)
  adding: content/AutogluonModels/ag-20260315_075816/models/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_075816/models/NeuralNetFastAI_BAG_L1/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_075816/models/NeuralNetFastAI_BAG_L1/S1F3/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_075816/models/NeuralNetFastAI_BAG_L1/S1F3/model.pkl (deflated 47%)
  adding: content/AutogluonModels/ag-20260315_075816/models/NeuralNetFastAI_BAG_L1/S1F2/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_07581